In [45]:
import importlib
import kernels.attention_batch_invariant as _attn_mod
import kernels.matmul_batch_invariant   as _mm_mod
import kernels.rmsnorm_batch_invariant  as _rms_mod
importlib.reload(_attn_mod)
importlib.reload(_mm_mod)
importlib.reload(_rms_mod)

from kernels.attention_batch_invariant import nki_attention_kernel_isa
from kernels.matmul_batch_invariant    import nki_matmul_kernel_isa
from kernels.rmsnorm_batch_invariant   import nki_rmsnorm_kernel_isa

In [46]:
import torch_xla
torch_xla.device()

device(type='xla', index=0)

In [47]:
import os
os.environ['NEURON_PLATFORM_TARGET_OVERRIDE'] = 'trn2'
os.environ['NEURON_CC_FLAGS'] = os.environ.get('NEURON_CC_FLAGS', '') + ' --cache_dir=/var/tmp/neuron-compile-cache'

# Batch Invariance — Full Kernel Test Suite

Covers all three NKI ISA kernels (MatMul, RMSNorm, Attention) plus a full
Transformer block forward pass and a vLLM-style continuous batching simulation.

Each test follows the same pattern as `test_determinism.ipynb`:
- **Run-to-run determinism**: same inputs, `deterministic=True` both calls, N iterations identical
- **Tile-size invariance**: `deterministic=True` vs `deterministic=False` on same inputs
  - `bfloat16` → `diff=0.0` (invariant — the main finding)
  - `float32`  → `diff!=0`  (not invariant — expected, documents the mechanism)

All inputs use `linspace(-1, 1)` matching `test_determinism.ipynb`.

---
# 1. MatMul Kernel

## 1a. Run-to-run determinism

In [48]:
import torch
import torch_xla.core.xla_model as xm

device = xm.xla_device()

def test_run_to_run(kernel_fn, inputs_fn, deterministic=True, iterations=10, label=''):
    """Run kernel N times with deterministic=True. All outputs must be bitwise identical."""
    args = inputs_fn()
    ref = kernel_fn(*args, deterministic=True)
    xm.mark_step()
    for i in range(iterations - 1):
        result = kernel_fn(*args, deterministic=True)
        xm.mark_step()
        max_diff = (result - ref).abs().max().item()
        if max_diff != 0:
            print(f'  {label} FAILED at iteration {i}: max_diff={max_diff}')
            return False
    print(f'  {label} PASSED: {iterations} iterations identical')
    return True


def test_tile_invariance(kernel_fn, inputs_fn, dtype, deterministic, label=''):
    """Compare det=True (larger tile) vs det=False (smaller tile).
    bfloat16 -> diff=0.0 (invariant). float32 -> diff!=0 (expected)."""
    # Create CPU tensors then move to device — same as test_tile_invariance.py
    cpu_args = inputs_fn(dtype, on_device=False)
    args = [t.to(device) for t in cpu_args]
    out_det = kernel_fn(*args, deterministic=True)
    xm.mark_step()
    out_nondet = kernel_fn(*args, deterministic=deterministic)
    xm.mark_step()
    diff = (out_det.cpu().float() - out_nondet.cpu().float()).abs().max().item()
    return {'label': label, 'dtype': str(dtype), 'diff': diff, 'invariant': diff == 0.0}


/tmp/ipykernel_1091092/3807403289.py:4: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


In [49]:
K, M, N = 512, 256, 512

def matmul_inputs(dtype=torch.bfloat16, on_device=True):
    a = torch.linspace(-1, 1, K * M, dtype=dtype).reshape(K, M)
    b = torch.linspace(-1, 1, K * N, dtype=dtype).reshape(K, N)
    if on_device:
        return a.to(device), b.to(device)
    return a, b

test_run_to_run(nki_matmul_kernel_isa,
                lambda: matmul_inputs(torch.bfloat16, on_device=True),
                iterations=10, label='matmul bfloat16')


/tmp/ipykernel_1091092/3807403289.py:10: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:57:54.000108:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_4128626200314693574+fad94d7c.hlo_module.pb
  matmul bfloat16 PASSED: 10 iterations identical


/tmp/ipykernel_1091092/3807403289.py:13: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


True

## 1b. Tile-size invariance — bfloat16

In [50]:
# deterministic=True both calls (baseline: same config)
test_tile_invariance(nki_matmul_kernel_isa, matmul_inputs, torch.bfloat16, label='matmul det/det', deterministic=True)

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'matmul det/det',
 'dtype': 'torch.bfloat16',
 'diff': 0.0,
 'invariant': True}

In [51]:
# deterministic=True vs False (K_TILE=128 vs 64)
test_tile_invariance(nki_matmul_kernel_isa, matmul_inputs, torch.bfloat16, deterministic=False, label='matmul det/nondet bfloat16')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'matmul det/nondet bfloat16',
 'dtype': 'torch.bfloat16',
 'diff': 0.0,
 'invariant': True}

## 1c. Tile-size invariance — float32 (variance expected)

In [52]:
test_tile_invariance(nki_matmul_kernel_isa, matmul_inputs, torch.float32, deterministic=True, label='matmul det/det float32')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'matmul det/det float32',
 'dtype': 'torch.float32',
 'diff': 0.0,
 'invariant': True}

In [53]:
test_tile_invariance(nki_matmul_kernel_isa, matmul_inputs, torch.float32, deterministic=False, label='matmul det/nondet float32')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'matmul det/nondet float32',
 'dtype': 'torch.float32',
 'diff': 6.103515625e-05,
 'invariant': False}

---
# 2. RMSNorm Kernel

## 2a. Run-to-run determinism

In [54]:
batch, hidden = 128, 512

def rmsnorm_inputs(dtype=torch.bfloat16, on_device=True):
    a = torch.linspace(-1, 1, batch * hidden, dtype=dtype).reshape(batch, hidden)
    g = torch.ones(hidden, dtype=dtype)
    if on_device:
        return a.to(device), g.to(device)
    return a, g

test_run_to_run(nki_rmsnorm_kernel_isa,
                lambda: rmsnorm_inputs(torch.bfloat16, on_device=True),
                iterations=10, label='rmsnorm bfloat16')


  rmsnorm bfloat16 PASSED: 10 iterations identical


/tmp/ipykernel_1091092/3807403289.py:10: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:13: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


True

## 2b. Tile-size invariance — bfloat16

In [55]:
test_tile_invariance(nki_rmsnorm_kernel_isa, rmsnorm_inputs, torch.bfloat16, deterministic=True, label='rmsnorm det/det')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'rmsnorm det/det',
 'dtype': 'torch.bfloat16',
 'diff': 0.0,
 'invariant': True}

In [56]:
test_tile_invariance(nki_rmsnorm_kernel_isa, rmsnorm_inputs, torch.bfloat16, deterministic=False, label='rmsnorm det/nondet bfloat16')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'rmsnorm det/nondet bfloat16',
 'dtype': 'torch.bfloat16',
 'diff': 0.0,
 'invariant': True}

## 2c. Tile-size invariance — float32 (variance expected)

In [57]:
test_tile_invariance(nki_rmsnorm_kernel_isa, rmsnorm_inputs, torch.float32, deterministic=True, label='rmsnorm det/det float32')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'rmsnorm det/det float32',
 'dtype': 'torch.float32',
 'diff': 0.0,
 'invariant': True}

In [58]:
test_tile_invariance(nki_rmsnorm_kernel_isa, rmsnorm_inputs, torch.float32, deterministic=False, label='rmsnorm det/nondet float32')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'rmsnorm det/nondet float32',
 'dtype': 'torch.float32',
 'diff': 2.384185791015625e-07,
 'invariant': False}

---
# 3. Attention Kernel

Input layout: `[d_head, seq]` — matches the reference kernel from `nki_samples/tutorials/attention_fwd_performance`.

The invariance-relevant variable is `KV_TILE` (FMAX_MOVING): `512` vs `256`.
This controls how the KV sequence is tiled during the `exp`/`sum` softmax pass,
which feeds into the final `scores @ V` PSUM accumulation.

## 3a. Run-to-run determinism

In [59]:
d_head = 128
seq_q, seq_k = 512, 512

def attn_inputs(dtype=torch.bfloat16, on_device=True):
    # Layout: [seq, d_head] — matches nki_attention_kernel_isa signature
    q = torch.linspace(-1,   1,   seq_q * d_head, dtype=dtype).reshape(seq_q, d_head)
    k = torch.linspace(-1,   1,   seq_k * d_head, dtype=dtype).reshape(seq_k, d_head)
    v = torch.linspace(-1,   1,   seq_k * d_head, dtype=dtype).reshape(seq_k, d_head)
    if on_device:
        return q.to(device), k.to(device), v.to(device)
    return q, k, v

test_run_to_run(nki_attention_kernel_isa,
                lambda: attn_inputs(torch.bfloat16, on_device=True),
                iterations=10, label='attention bfloat16')


/tmp/ipykernel_1091092/3807403289.py:10: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


  attention bfloat16 PASSED: 10 iterations identical


/tmp/ipykernel_1091092/3807403289.py:13: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


True

## 3b. Tile-size invariance — bfloat16

In [60]:
# ── Raw attention diagnostic (no test harness) ──────────────────────────────
import importlib
import kernels.attention_batch_invariant as _attn_mod
importlib.reload(_attn_mod)
from kernels.attention_batch_invariant import nki_attention_kernel_isa as _attn

_q = torch.linspace(-1, 1, seq_q * d_head, dtype=torch.bfloat16).reshape(seq_q, d_head).to(device)
_k = torch.linspace(-1, 1, seq_k * d_head, dtype=torch.bfloat16).reshape(seq_k, d_head).to(device)
_v = torch.linspace(-1, 1, seq_k * d_head, dtype=torch.bfloat16).reshape(seq_k, d_head).to(device)

_out1 = _attn(_q, _k, _v, deterministic=True);  xm.mark_step()
_out2 = _attn(_q, _k, _v, deterministic=False); xm.mark_step()

_c1 = _out1.cpu().float(); _c2 = _out2.cpu().float()
print(f'det=True  : nan={_c1.isnan().sum().item()}  inf={_c1.isinf().sum().item()}  max={_c1.abs().max().item():.4f}')
print(f'det=False : nan={_c2.isnan().sum().item()}  inf={_c2.isinf().sum().item()}  max={_c2.abs().max().item():.4f}')
print(f'diff      : {(_c1 - _c2).abs().max().item()}')


/tmp/ipykernel_1091092/2980100706.py:11: DeprecationWarning: Use torch_xla.sync instead
  _out1 = _attn(_q, _k, _v, deterministic=True);  xm.mark_step()


det=True  : nan=0  inf=0  max=0.9141
det=False : nan=0  inf=0  max=0.9141
diff      : 0.0


/tmp/ipykernel_1091092/2980100706.py:12: DeprecationWarning: Use torch_xla.sync instead
  _out2 = _attn(_q, _k, _v, deterministic=False); xm.mark_step()


In [61]:
test_tile_invariance(nki_attention_kernel_isa, attn_inputs, torch.bfloat16, deterministic=True, label='attention det/det')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'attention det/det',
 'dtype': 'torch.bfloat16',
 'diff': 0.0,
 'invariant': True}

In [62]:
test_tile_invariance(nki_attention_kernel_isa, attn_inputs, torch.bfloat16, deterministic=False, label='attention det/nondet bfloat16')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'attention det/nondet bfloat16',
 'dtype': 'torch.bfloat16',
 'diff': 0.0,
 'invariant': True}

## 3c. Tile-size invariance — float32 (variance expected)

In [63]:
test_tile_invariance(nki_attention_kernel_isa, attn_inputs, torch.float32, deterministic=True, label='attention det/det float32')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'attention det/det float32',
 'dtype': 'torch.float32',
 'diff': 0.0,
 'invariant': True}

In [64]:
test_tile_invariance(nki_attention_kernel_isa, attn_inputs, torch.float32, deterministic=False, label='attention det/nondet float32')

/tmp/ipykernel_1091092/3807403289.py:29: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3807403289.py:31: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


{'label': 'attention det/nondet float32',
 'dtype': 'torch.float32',
 'diff': 3.5762786865234375e-07,
 'invariant': False}

---
# 4. Full Transformer Block Forward Pass

Block: `x → RMSNorm → QKV proj → Attention → out-proj + residual → RMSNorm → FFN → residual`

All sub-ops use the NKI ISA kernels. The `deterministic` flag is passed uniformly
to every kernel call. This tests whether the invariance property holds end-to-end
through a realistic compute graph.

**Weight methodology**: linspace weights are used so both matrix operands in
every matmul have linspace structure — the same condition that guarantees
bfloat16 products land on the same coarse grid regardless of tile grouping.
`scale=0.02` keeps intermediate activations well within bfloat16 range.

**Scope**: kernel-level invariance propagated through a block, not
serving-framework or model-level invariance.

In [65]:
def nki_transformer_block(x, weights, deterministic=True):
    device = x.device
    w = {k: v.to(device) for k, v in weights.items()}

    def mm(a, b):
        return nki_matmul_kernel_isa(a.T.contiguous(), b, deterministic=deterministic)

    def rms(a, g):
        return nki_rmsnorm_kernel_isa(a, g, deterministic=deterministic)

    def attn(q, k, v):
        return nki_attention_kernel_isa(q, k, v, deterministic=deterministic)

    x_norm   = rms(x, w['g_attn'])
    q        = mm(x_norm, w['wq'])
    k        = mm(x_norm, w['wk'])
    v        = mm(x_norm, w['wv'])
    attn_out = attn(q, k, v)
    x        = x + mm(attn_out, w['wo'])
    x_norm   = rms(x, w['g_ffn'])
    h        = mm(x_norm, w['w1'])
    h        = torch.relu(h)
    x        = x + mm(h, w['w2'])
    return x


def make_block_weights(d_model, d_head, d_ffn, dtype):
    def linspace_w(fan_in, fan_out, scale=0.02):
        return torch.linspace(-scale, scale, fan_in * fan_out,
                              dtype=dtype).reshape(fan_in, fan_out)
    return {
        'wq':    linspace_w(d_model, d_head),
        'wk':    linspace_w(d_model, d_head),
        'wv':    linspace_w(d_model, d_head),
        'wo':    linspace_w(d_head,  d_model),
        'w1':    linspace_w(d_model, d_ffn),
        'w2':    linspace_w(d_ffn,   d_model),
        'g_attn': torch.ones(d_model, dtype=dtype),
        'g_ffn':  torch.ones(d_model, dtype=dtype),
    }

print('transformer block helpers defined')

transformer block helpers defined


## 4a. Run-to-run determinism — full block

In [66]:
seq, d_model, d_head, d_ffn = 512, 256, 128, 512
iterations = 100

for dtype in [torch.bfloat16, torch.float32]:
    x_cpu = torch.linspace(-1, 1, seq * d_model, dtype=dtype).reshape(seq, d_model)
    x = x_cpu.to(device)
    weights = make_block_weights(d_model, d_head, d_ffn, dtype)

    ref = nki_transformer_block(x, weights, deterministic=True)
    xm.mark_step()
    ref_f = ref.cpu().float()
    max_diff = 0.0
    for _ in range(iterations - 1):
        out = nki_transformer_block(x, weights, deterministic=True)
        xm.mark_step()
        max_diff = max(max_diff, (out.cpu().float() - ref_f).abs().max().item())

    status = 'PASSED' if max_diff == 0.0 else f'FAILED (max_diff={max_diff:.3e})'
    print(f'  forward pass {str(dtype):20s} {iterations} runs: {status}')


/tmp/ipykernel_1091092/3626422624.py:10: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:57:57.000712:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_8310631996945747722+fad94d7c.hlo_module.pb


/tmp/ipykernel_1091092/3626422624.py:15: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


  forward pass torch.bfloat16       100 runs: PASSED


/tmp/ipykernel_1091092/3626422624.py:10: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:00.000621:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_18419651769883600515+fad94d7c.hlo_module.pb


/tmp/ipykernel_1091092/3626422624.py:15: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


  forward pass torch.float32        100 runs: PASSED


## 4b. Tile-size invariance — full block

In [67]:
for dtype in [torch.bfloat16, torch.float32]:
    x_cpu = torch.linspace(-1, 1, seq * d_model, dtype=dtype).reshape(seq, d_model)
    x = x_cpu.to(device)
    weights = make_block_weights(d_model, d_head, d_ffn, dtype)

    out_det = nki_transformer_block(x, weights, deterministic=True)
    xm.mark_step()   # force execution before second block call
    out_nondet = nki_transformer_block(x, weights, deterministic=False)
    xm.mark_step()
    diff = (out_det.cpu().float() - out_nondet.cpu().float()).abs().max().item()

    expected = dtype == torch.bfloat16
    status = 'PASS' if (diff == 0.0) == expected else f'FAIL diff={diff:.3e}'
    note = '' if expected else f'  (diff={diff:.2e}, variance expected for float32)'
    print(f'  block det/nondet {str(dtype):20s}: {status}{note}')


/tmp/ipykernel_1091092/58765706.py:7: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # force execution before second block call
/tmp/ipykernel_1091092/58765706.py:9: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:03.000699:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_4318687694982088872+fad94d7c.hlo_module.pb
  block det/nondet torch.bfloat16      : PASS


/tmp/ipykernel_1091092/58765706.py:7: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # force execution before second block call
/tmp/ipykernel_1091092/58765706.py:9: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:06.000033:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_2048034649362581182+fad94d7c.hlo_module.pb
  block det/nondet torch.float32       : PASS  (diff=1.96e-06, variance expected for float32)


---
# 5. Continuous Batching Simulation (vLLM-style)

vLLM packs variable-length requests into a fixed batch — changing tile counts
between iterations. A request's output must not change based on what other
requests are packed alongside it.

Three scenarios:
- **Position independence**: target row at different positions in the batch
- **Neighbor independence**: same target, different co-packed sequences
- **KV-context length**: attention with varying `seq_k` (different tile counts)

In [68]:
print('--- Continuous Batching: RMSNorm position independence ---')
print('Target row at positions 0, 1, 63, 127 in a batch of 128\n')

hidden = 512
batch_size = 128

for dtype in [torch.bfloat16, torch.float32]:
    g_cpu      = torch.ones(hidden, dtype=dtype)
    target_cpu = torch.linspace(-1, 1, hidden, dtype=dtype)
    noise_cpu  = torch.linspace(-0.5, 0.5, batch_size * hidden, dtype=dtype).reshape(batch_size, hidden)
    g = g_cpu.to(device)

    x_ref = noise_cpu.clone(); x_ref[0] = target_cpu
    ref_row_out = nki_rmsnorm_kernel_isa(x_ref.to(device), g, deterministic=True)
    xm.mark_step()
    ref_row = ref_row_out[0].cpu().float()

    for pos in [0, 1, 63, 127]:
        x = noise_cpu.clone(); x[pos] = target_cpu
        out = nki_rmsnorm_kernel_isa(x.to(device), g, deterministic=True)
        xm.mark_step()
        diff = (out[pos].cpu().float() - ref_row).abs().max().item()
        print(f'  dtype={str(dtype):20s} pos={pos:3d}: {"PASS" if diff == 0.0 else f"FAIL diff={diff:.3e}"}')
    print()


--- Continuous Batching: RMSNorm position independence ---
Target row at positions 0, 1, 63, 127 in a batch of 128



/tmp/ipykernel_1091092/2844995821.py:15: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:07.000280:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_10172690520440305496+fad94d7c.hlo_module.pb
  dtype=torch.bfloat16       pos=  0: PASS


/tmp/ipykernel_1091092/2844995821.py:21: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:08.000552:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_1144929939325208945+fad94d7c.hlo_module.pb
  dtype=torch.bfloat16       pos=  1: PASS


...

.
Compiler status PASS
2026-05-04 17:58:13.000618:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_13811828888347625614+fad94d7c.hlo_module.pb
  dtype=torch.float32        pos=  1: PASS


/tmp/ipykernel_1091092/2844995821.py:21: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:14.000910:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_8101497411674931400+fad94d7c.hlo_module.pb
  dtype=torch.float32        pos= 63: PASS


/tmp/ipykernel_1091092/2844995821.py:21: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:16.000189:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_9836409639101051339+fad94d7c.hlo_module.pb
  dtype=torch.float32        pos=127: PASS



In [69]:
print('--- Continuous Batching: RMSNorm neighbor independence ---')
print('Same target row, 3 different neighbor sets — output must be identical\n')

for dtype in [torch.bfloat16, torch.float32]:
    g_cpu = torch.ones(hidden, dtype=dtype)
    g = g_cpu.to(device)
    target_cpu = torch.linspace(-1, 1, hidden, dtype=dtype)
    outputs = []
    for seed in [0, 1, 2]:
        torch.manual_seed(seed)
        x_cpu = torch.randn(batch_size, hidden, dtype=dtype)
        x_cpu[0] = target_cpu
        out = nki_rmsnorm_kernel_isa(x_cpu.to(device), g, deterministic=True)
        xm.mark_step()
        outputs.append(out[0].cpu().float())

    d01 = (outputs[0] - outputs[1]).abs().max().item()
    d02 = (outputs[0] - outputs[2]).abs().max().item()
    ok  = d01 == 0.0 and d02 == 0.0
    print(f'  dtype={str(dtype):20s} neighbor-independent: {"PASS" if ok else f"FAIL d01={d01:.3e} d02={d02:.3e}"}')
print()


--- Continuous Batching: RMSNorm neighbor independence ---
Same target row, 3 different neighbor sets — output must be identical

  dtype=torch.bfloat16       neighbor-independent: PASS
  dtype=torch.float32        neighbor-independent: PASS



/tmp/ipykernel_1091092/3488558045.py:14: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


In [70]:
print('--- Continuous Batching: Attention — varying seq_k ---')
print('Same Q/K/V content, seq_k=512: det/det identical; det/nondet bfloat16=0 float32!=0\n')

d_head_attn, seq_q_attn, seq_k_attn = 128, 512, 512

for dtype in [torch.bfloat16, torch.float32]:
    q_cpu = torch.linspace(-1, 1,     seq_q_attn * d_head_attn, dtype=dtype).reshape(seq_q_attn, d_head_attn)
    k_cpu = torch.linspace(-1, 1,     seq_k_attn * d_head_attn, dtype=dtype).reshape(seq_k_attn, d_head_attn)
    v_cpu = torch.linspace(-0.5, 0.5, seq_k_attn * d_head_attn, dtype=dtype).reshape(seq_k_attn, d_head_attn)
    q, k, v = q_cpu.to(device), k_cpu.to(device), v_cpu.to(device)

    out_det1 = nki_attention_kernel_isa(q, k, v, deterministic=True)
    xm.mark_step()
    out_det2 = nki_attention_kernel_isa(q, k, v, deterministic=True)
    xm.mark_step()
    out_nondet = nki_attention_kernel_isa(q, k, v, deterministic=False)
    xm.mark_step()

    diff_det    = (out_det1.cpu().float() - out_det2.cpu().float()).abs().max().item()
    diff_nondet = (out_det1.cpu().float() - out_nondet.cpu().float()).abs().max().item()

    expected = dtype == torch.bfloat16
    det_ok    = diff_det == 0.0
    nondet_ok = diff_nondet == 0.0 if expected else diff_nondet > 0.0

    print(f'  dtype={str(dtype):20s}'
          f'  det/det={diff_det:.2e} {"PASS" if det_ok else "FAIL"}'
          f'  det/nondet={diff_nondet:.2e} {"PASS" if nondet_ok else "FAIL"}'
          f'{"" if expected else "  (variance expected)"}')
print()


--- Continuous Batching: Attention — varying seq_k ---
Same Q/K/V content, seq_k=512: det/det identical; det/nondet bfloat16=0 float32!=0

  dtype=torch.bfloat16        det/det=0.00e+00 PASS  det/nondet=0.00e+00 PASS
  dtype=torch.float32         det/det=0.00e+00 PASS  det/nondet=1.79e-07 PASS  (variance expected)



/tmp/ipykernel_1091092/1330232453.py:13: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/1330232453.py:15: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/1330232453.py:17: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


In [71]:
print('--- Continuous Batching: Full Block ---')
print('Same sequence, det/det must be identical; det/nondet bfloat16=0 float32!=0\n')

seq_cb, d_model_cb, d_head_cb, d_ffn_cb = 512, 128, 128, 256

for dtype in [torch.bfloat16, torch.float32]:
    x_cpu = torch.linspace(-1, 1, seq_cb * d_model_cb, dtype=dtype).reshape(seq_cb, d_model_cb)
    x = x_cpu.to(device)
    w = make_block_weights(d_model_cb, d_head_cb, d_ffn_cb, dtype)

    ref = nki_transformer_block(x, w, deterministic=True)
    xm.mark_step()

    out_det2 = nki_transformer_block(x, w, deterministic=True)
    xm.mark_step()
    diff_det = (ref.cpu().float() - out_det2.cpu().float()).abs().max().item()

    out_nondet = nki_transformer_block(x, w, deterministic=False)
    xm.mark_step()
    diff_nondet = (ref.cpu().float() - out_nondet.cpu().float()).abs().max().item()

    expected = dtype == torch.bfloat16
    det_ok    = diff_det == 0.0
    nondet_ok = diff_nondet == 0.0 if expected else diff_nondet > 0.0

    print(f'  dtype={str(dtype):20s}'
          f'  det/det={diff_det:.2e} {"PASS" if det_ok else "FAIL"}'
          f'  det/nondet={diff_nondet:.2e} {"PASS" if nondet_ok else "FAIL"}'
          f'{"" if expected else "  (variance expected)"}')
print()


--- Continuous Batching: Full Block ---
Same sequence, det/det must be identical; det/nondet bfloat16=0 float32!=0



/tmp/ipykernel_1091092/3777649639.py:12: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:18.000141:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_9736336853906890484+fad94d7c.hlo_module.pb


/tmp/ipykernel_1091092/3777649639.py:15: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3777649639.py:19: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:20.000105:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_14548041311692474700+fad94d7c.hlo_module.pb
  dtype=torch.bfloat16        det/det=0.00e+00 PASS  det/nondet=0.00e+00 PASS


/tmp/ipykernel_1091092/3777649639.py:12: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:22.000009:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_5886331490924844791+fad94d7c.hlo_module.pb


/tmp/ipykernel_1091092/3777649639.py:15: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()
/tmp/ipykernel_1091092/3777649639.py:19: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


.
Compiler status PASS
2026-05-04 17:58:23.000983:  1091092  [INFO]: Compilation Successfully Completed for model.MODULE_16064713817923054692+fad94d7c.hlo_module.pb
  dtype=torch.float32         det/det=0.00e+00 PASS  det/nondet=4.95e-07 PASS  (variance expected)



---
# Summary

| Kernel | dtype | det/det | det/nondet | Expected |
|---|---|---|---|---|
| MatMul | bfloat16 | 0.0 | 0.0 | invariant |
| MatMul | float32  | 0.0 | ~6e-05 | not invariant |
| RMSNorm | bfloat16 | 0.0 | 0.0 | invariant |
| RMSNorm | float32  | 0.0 | ~2e-07 | not invariant |
| Attention | bfloat16 | 0.0 | 0.0 | invariant |
| Attention | float32  | 0.0 | ~3e-07 | not invariant |
| Forward block | bfloat16 | 0.0 | 0.0 | invariant |
| Forward block | float32  | 0.0 | ~2e-06 | not invariant |

**Key finding**: bfloat16's 7-bit mantissa snaps every multiply result to a coarse grid
before it enters the float32 PSUM — so no matter how the K/KV dimension is tiled,
the inputs to the accumulator are identical. Batch invariance is free for bfloat16
on NeuronCore given normalized input distributions.

**Scope**: this result is scoped to these NKI ISA kernels operating in bfloat16.
It is not a claim about model-level or serving-framework-level batch invariance.
Each kernel in a model's compute graph must independently satisfy this constraint.